In [1]:
%run std_methods.ipynb

In [2]:
import pandas as pd, math, datetime

In [3]:
logger =  initialize_logging()

In [4]:
def generate_date_sequence(start_date_YYYYMMDD,end_date_YYYYMMDD,period):
    
    period = period.lower()
    start_date = datetime.datetime.strptime(start_date_YYYYMMDD,'%Y-%m-%d')
    end_date = datetime.datetime.strptime(end_date_YYYYMMDD,'%Y-%m-%d')
    
    if period == "monthly":
        first_of_starting_month=start_date.strftime('%Y-%m-')+"01"
        date_offset=start_date-datetime.datetime.strptime(first_of_starting_month,'%Y-%m-%d')
        relevant_month_starts =  pd.date_range(start=first_of_starting_month, end=end_date, freq='MS').to_pydatetime().tolist()
        return [ dt + date_offset for dt in relevant_month_starts ]

    if period == "daily":
        return pd.date_range(start=start_date, end=end_date, freq='D').to_pydatetime().tolist()

    if period == "90 days":
        number_of_90_day_periods = math.floor((end_date - start_date).days / 90)
        return [start_date + datetime.timedelta(days=offset_index*90) for offset_index in range(0, number_of_90_day_periods)]

    if period == "biweekly":
        number_of_14_day_periods = math.floor((end_date - start_date).days / 14)
        return [start_date + datetime.timedelta(days=offset_index*14) for offset_index in range(0, number_of_14_day_periods)]

def generate_series_from_item_schedule(budget_data,accounts_df,start_date_YYYYMMDD,num_days):

    start_date = datetime.datetime.strptime(start_date_YYYYMMDD,'%Y-%m-%d')
    end_date = start_date + datetime.timedelta(days=num_days)
    end_date_YYYYMMDD = end_date.strftime('%Y-%m-%d')
    
    account_name_to_type_map = get_account_name_to_type_map(accounts_df)

    #Monthly
    monthly_items_df=budget_data.loc[budget_data['Cadence'] == "Monthly"]
    monthly_items_list=[]
    for index, row in monthly_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_monthly_sequence = generate_date_sequence(current_start_date_YYYYMMDD,end_date_YYYYMMDD,"monthly")
    
        for dt in current_monthly_sequence:
            monthly_items_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])
            

    #Daily
    daily_items_df=budget_data.loc[budget_data['Cadence'] == "Daily"]
    daily_items_list=[]
    for index, row in daily_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]
        
        current_daily_sequence = generate_date_sequence(current_start_date_YYYYMMDD, end_date_YYYYMMDD, "daily")
 
        for dt in current_daily_sequence:            
            daily_items_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])
                    

    #Bi-weekly
    biweekly_items_df = budget_data.loc[budget_data['Cadence'] == "Biweekly"]
    biweekly_items_list = []
    for index, row in biweekly_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_biweekly_sequence = generate_date_sequence(current_start_date_YYYYMMDD, end_date_YYYYMMDD, "biweekly")
  
        for dt in current_biweekly_sequence:
            biweekly_items_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])
                    

    #Once
    once_items_df=budget_data.loc[budget_data['Cadence'] == "Once"]
    once_items_list=[]
    for index, row in once_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]
  
        once_items_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])
                

    #Every 3 months
    every_3_months_items_df=budget_data.loc[budget_data['Cadence'] == "Every 3 months"]
    every_3_months_list=[]
    for index, row in every_3_months_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_90_day_sequence = generate_date_sequence(current_start_date_YYYYMMDD, end_date_YYYYMMDD, "90 days")
        
        for dt in current_90_day_sequence:
            every_3_months_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])

    scheduled_items_df = pd.DataFrame(monthly_items_list + daily_items_list + every_3_months_list + once_items_list + biweekly_items_list)
    scheduled_items_df.columns = ["Date","Amount","Account From","Account To","Priority","Memo"]
    scheduled_items_df = scheduled_items_df[scheduled_items_df.Date >= start_date]
    return scheduled_items_df.sort_values(by="Date")


def validate_parameter_tables(budget_items_df, 
                              priority_account_rules_df, 
                              priority_item_rules_df, 
                              accounts_df, 
                              milestones_df):
    
    #Since i dont know how to use great ewxpectations all the way, this data type casting will not be reflected in great expectations tests
    budget_items_df['Start Date'] = pd.to_datetime(budget_items_df['Start Date'], format='%Y-%m-%d')
    
    
    #save dfs separately for great_expectations
    budget_items_df.to_csv('/home/hdickie/sandbox/budget_items.csv')
    priority_account_rules_df.to_csv('/home/hdickie/sandbox/priority_account_rules.csv')
    priority_item_rules_df.to_csv('/home/hdickie/sandbox/priority_item_rules.csv')
    accounts_df.to_csv('/home/hdickie/sandbox/accounts.csv')
    milestones_df.to_csv('/home/hdickie/sandbox/milestones.csv')
    
    error_flag = False
    
    ###By sheet:
    
    # Budget Items
    data_source_name = "expense_forecast_budget_items"
    data_source_path = '/home/hdickie/sandbox/budget_items.csv'
    data_asset_name="expense_forecast_budget_items"
    expectation_suite_name="expense_forecast"
    checkpoint_name="validate_parameter_tables"
    context, validator = getValidatorAndUpdatedContext(data_source_name, data_source_path, data_asset_name, expectation_suite_name, checkpoint_name)
    
    #assert table schema
    error_flag = error_flag and not validator.expect_column_to_exist('Start Date').success #these method calls have side effects 
    error_flag = error_flag and not validator.expect_column_to_exist('Priority').success
    error_flag = error_flag and not validator.expect_column_to_exist('Cadence').success
    error_flag = error_flag and not validator.expect_column_to_exist('Amount').success
    error_flag = error_flag and not validator.expect_column_to_exist('Paid From').success
    error_flag = error_flag and not validator.expect_column_to_exist('Paid To').success
    error_flag = error_flag and not validator.expect_column_to_exist('Memo').success
    
    #assert non-nulls
    error_flag = error_flag and not validator.expect_column_values_to_not_be_null('Priority').success
    error_flag = error_flag and not validator.expect_column_values_to_not_be_null('Cadence').success
    error_flag = error_flag and not validator.expect_column_values_to_not_be_null('Amount').success
    
    # assert that all start dates are valid
    error_flag = error_flag and not validator.expect_column_values_to_match_regex('Start Date','[0-9]{4}-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])').success
    
    # assert that all priorities have defined behavior
    # TODO: if new priority levels become athign this will need to be adjusted
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Cadence',[1,2,3,4,5,6]).success
    
    # assert that cadence values are in set
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Cadence',['once','daily','weekly','monthly','biweekly','yearly','Once','Daily','Weekly','Monthly','Biweekly','Yearly']).success
    
    # assert that amounts are valid floats
    error_flag = error_flag and not validator.expect_column_values_to_be_of_type('Amount','float').success
    
    # assert that paid from/to values are in set
    #TODO: New accounts will need to be added here, possibly dynamically
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Paid From',['Checking','Credit 1','Savings']).success
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Paid To',['Checking','Credit 1','Savings']).success
    
    #runValidation(context,validator,checkpoint_name,budget_items_df)
    
    # Priority Account Rules
    #assert that all priority indices are integers (this is the source of truth column)
    #assert that account names are in set
    # assert Balance LB/UB are valid floats or +/-Inf
    
    # Priority Item Rules
    #would only need to validate if we were defining new priroity levels which is otuside of scope 
    
    # Accounts
    #assert balances columns are valid floats or +/-Inf
    #assert interest rate is valid float
    #assert interest type is Simple or Compound
    #assert billing_start_dt is valid dt
    
    # Milestones
    #assert account names are in set
    #assert min/max balacne are valid floats or +/-Inf
    
    if error_flag:
        logger.info("Errors were detected. Please check validation docs at file:///home/hdickie/great_expectations/uncommitted/data_docs/local_site/index.html")
    
    return error_flag

def read_budget_item_excel_sheet(excel_path,validate=False):
    budget_items_df = pd.read_excel(excel_path,sheet_name = 'Budget_Items')
    priority_account_rules_df = pd.read_excel(excel_path,sheet_name = 'Priority_Account_Rules')
    priority_item_rules_df = pd.read_excel(excel_path,sheet_name = 'Priority_Item_Rules')
    accounts_df = pd.read_excel(excel_path,sheet_name = 'Accounts')
    milestones_df = pd.read_excel(excel_path,sheet_name = 'Milestones')
    
    if validate:
        validate_parameter_tables(budget_items_df, priority_account_rules_df, priority_item_rules_df, accounts_df, milestones_df)
    
    return [budget_items_df, priority_account_rules_df, priority_item_rules_df, accounts_df, milestones_df]
    
def write_results(results):
    results.to_csv('/home/hdickie/sandbox/results.csv')
    
def get_initial_balances(accounts_df):
    acct_balances = {}
    for index,row in accounts_df.iterrows():
        acct_balances[row['Account Name']] = row['Current Balance']
    return acct_balances

def get_account_name_to_type_map(accounts_df):
    acct_types = {}
    for index,row in accounts_df.iterrows():
        acct_types[row['Account Name']] = row['Account Type']
    return acct_types


In [74]:
def satisfice(start_date_YYYYMMDD,initial_balances_dict,scheduled_items_df,accounts_df,days_out):
    logger.info('enter satisfice()')
    logger.info('start_date_YYYYMMDD:'+str(start_date_YYYYMMDD))
    logger.info('initial_balances_dict:'+str(initial_balances_dict))
    logger.info('days_out:'+str(days_out))
    account_name_to_type_map = get_account_name_to_type_map(accounts_df)
    
    start_date = datetime.datetime.strptime(start_date_YYYYMMDD,'%Y-%m-%d')
    
    #create first row
    initial_row_names = ['Date']
    for key in initial_balances_dict.keys():
        initial_row_names += [key]
    initial_row_names += ['Memo']

    initial_row = [start_date]
    for val in initial_balances_dict.values():
        initial_row += [val]
    initial_row += ['']

    initial_row_df = pd.DataFrame(initial_row).T
    initial_row_df.columns = initial_row_names

    all_days = [(start_date + datetime.timedelta(days=d)).strftime('%Y-%m-%d') for d in range(0,days_out)]
    
    loan_acct_names = []
    for acct_name in accounts_df['Account Name']:
        if 'Loan' in acct_name and 'Interest' not in acct_name:
            loan_acct_names += [acct_name]
            
    loan_account_name_to_interest_rate_map = {}
    for acct_name in loan_acct_names:
        loan_account_name_to_interest_rate_map[acct_name] = accounts_df[accounts_df['Account Name'] == acct_name]['Interest Rate APR'].iat[0]
    
    #credit_acct_name_billing_dt_tuples_list = []
    billing_dt_to_credit_acct_name_list_map = {}
    for index, row in accounts_df.iterrows():
        if 'Credit' in row['Account Name'] and 'Grace Period Balance' not in row['Account Name']:
            cc_start_date_YYYYMMDD = row['Billing Start Dt']
            cc_end_date_YYYYMMDD = (datetime.datetime.strptime(start_date_YYYYMMDD,'%Y-%m-%d') + datetime.timedelta(days=days_out)).strftime('%Y-%m-%d')

            logger.info('cc_start_date_YYYYMMDD:'+str(cc_start_date_YYYYMMDD))
            logger.info('cc_end_date_YYYYMMDD:'+str(cc_end_date_YYYYMMDD))
            for dt in generate_date_sequence( cc_start_date_YYYYMMDD, cc_end_date_YYYYMMDD,'monthly'):
                #credit_acct_name_billing_dt_tuples_list += [(row['Account Name'],dt)]
                
                #logger.info(str(min(all_days))+' ?<= '+str(dt)+' ?<= '+str(max(all_days))+' : '+str(min(all_days) <= dt.strftime('%Y-%m-%d') and dt.strftime('%Y-%m-%d') <= max(all_days)))
                if min(all_days) <= dt.strftime('%Y-%m-%d') and dt.strftime('%Y-%m-%d') <= max(all_days):
                    pass
                else:
                    continue
                
                if dt.strftime('%Y-%m-%d') in billing_dt_to_credit_acct_name_list_map.keys():
                    billing_dt_to_credit_acct_name_list_map[dt.strftime('%Y-%m-%d') ] += [row['Account Name']]
                else:
                    billing_dt_to_credit_acct_name_list_map[dt.strftime('%Y-%m-%d') ] = [row['Account Name']]

    for d in all_days:
        #logger.info('start_date:'+str(start_date))
        #logger.info('d ?= start_date_YYYYMMDD:'+str(d == start_date_YYYYMMDD))
        logger.info('d:'+str(d))
        if d == start_date_YYYYMMDD:
            previous_rows_df = initial_row_df
            continue            

        new_row_df = previous_rows_df.tail(1).copy()
        new_row_df.loc[0,"Date"] = d
        
        #logger.info('billing_dt_to_credit_acct_name_list_map.keys():'+str(billing_dt_to_credit_acct_name_list_map.keys()))
        if d in billing_dt_to_credit_acct_name_list_map.keys():
 
            #credit txns are first put in Grace Period Balance.
            #then, the first time through this loop, any balance gets moved to "Credit"
            #then, the second time though this loop, any balance in Credit will accrue interest
            
            grace_period_balance = new_row_df.loc[0,billing_dt_to_credit_acct_name_list_map[d][0]+' Grace Period Balance']
            interest_accruing_balance = new_row_df.loc[0,billing_dt_to_credit_acct_name_list_map[d][0]]
            
            accrued_interest = interest_accruing_balance * 0.2374/12 #TODO update this if i ever have multiple credit cards
            new_interest_accruing_balance = grace_period_balance + interest_accruing_balance + accrued_interest
            
            #todo: not totally sure this is right
            new_row_df.loc[0,billing_dt_to_credit_acct_name_list_map[d][0]+' Grace Period Balance'] = 0
            new_row_df.loc[0,billing_dt_to_credit_acct_name_list_map[d][0]] = new_interest_accruing_balance
            
            logger.info('GRACE PERIOD LOGIC')
            logger.info('Grace Period Balance.........:'+str(grace_period_balance))
            logger.info('Interest Accruing Balance....:'+str(interest_accruing_balance))
            logger.info('Accrued Interest.............:'+str(accrued_interest))
            logger.info('New Interest Accruing Balance:'+str(new_interest_accruing_balance))

            
        
        #do daily interest accruals for this day
        for acct_name in loan_acct_names:
            old_interest_value = previous_rows_df.tail(1)[acct_name+" Interest"]
            old_principal_value = previous_rows_df.tail(1)[acct_name]
            
            new_marginal_interest = (old_principal_value*(loan_account_name_to_interest_rate_map[acct_name]/365.25)).iat[0]
            #logger.info('new_marginal_interest:'+str(new_marginal_interest))
            new_row_df.iloc[:,new_row_df.columns == (acct_name+' Interest')] += new_marginal_interest
            

        relevant_items_df = scheduled_items_df[ (scheduled_items_df['Date'] == d) & (scheduled_items_df['Priority'] == 1)]
        
        logger.debug('relevant_items_df:')
        logger.debug(relevant_items_df.to_string())
        for index, row in relevant_items_df.iterrows():
            assert sum(new_row_df.columns == row["Account From"]) == 1 or pd.isna(row["Account From"])
            assert sum(new_row_df.columns == row["Account To"]) == 1 or pd.isna(row["Account To"])
            
            row.Amount = float(row.Amount)
            logger.info('row.Amount:'+str(row.Amount))
            
            if not pd.isna(row["Account To"]):
                account_to__type = account_name_to_type_map[row["Account To"]]
            else:
                account_to__type = ""
                
            if not pd.isna(row["Account From"]):
                account_from__type = account_name_to_type_map[row["Account From"]]
            else:
                account_from__type = ""
            
            #Loan Payments, from any account
            if account_to__type == 'Loan':
                old_principal_value = float(new_row_df.loc[:,new_row_df.columns == row["Account To"]].iat[0,0])
                old_interest_value = float(new_row_df.loc[:,new_row_df.columns == (row["Account To"]+' Interest')].iat[0,0])

                #payment goes to interest only
                #logger.info('row.Amount*-1:'+str(row.Amount*-1)+' ?= '+str(old_interest_value))
                if row.Amount <= old_interest_value:
                    #principal remains unchanged
                    new_principal_value = old_principal_value
                    new_row_df.loc[:,new_row_df.columns == row["Account To"]]  = new_principal_value

                    new_interest_value = old_interest_value - row.Amount
                    new_row_df.loc[:,new_row_df.columns == (row["Account To"]+' Interest')] = new_interest_value

                #this includes the case where interest = 0
                elif row.Amount > old_interest_value:

                    new_principal_value = old_principal_value + old_interest_value - row.Amount
                    new_row_df.loc[:,new_row_df.columns == row["Account To"]] = new_principal_value

                    #Interest is now 0
                    new_interest_value = 0
                    new_row_df.loc[:,new_row_df.columns == (row["Account To"]+' Interest')] = new_interest_value

                else:
                    logger.error('error in satisfice::loan interest payment calculation')
                    
                new_row_df.loc[:,new_row_df.columns == (row["Account From"])] = new_row_df.loc[:,new_row_df.columns == (row["Account From"])] - row.Amount
                    
            elif account_from__type == 'Credit':
                logger.debug('Case: FROM CREDIT')
                old_value = float(new_row_df.loc[:,new_row_df.columns == (row["Account From"] + ' Grace Period Balance')].iat[0,0])
                new_value = old_value + float(row.Amount)

                logger.info(str(d)+' '+str(index)+' '+str(row.Memo)+' ; '+str(row["Account From"] + ' Grace Period Balance')+' '+str(float(old_value))+' ('+str(float(row.Amount))+') -> '+str(float(new_value)))

                new_row_df.loc[:,new_row_df.columns == (row["Account From"] + ' Grace Period Balance')] = new_value
                
                if account_to__type != '':
                    pass # i dont think this ever happens
                    raise NotImplementedError
                    
            elif account_from__type == 'Checking' and account_to__type == 'Credit':
                old_from_value = float(new_row_df.loc[:,new_row_df.columns == (row["Account From"])].iat[0,0])
                new_from_value = old_from_value - float(row.Amount)
                
                #could put a log statement here
                
                new_row_df.loc[:,new_row_df.columns == (row["Account From"])] = new_from_value
                
                #implement credit card payment
                old_grace_period_balance_value = new_row_df.loc[:,new_row_df.columns == (row["Account To"] + ' Grace Period Balance')].iat[0,0]
                old_interest_accruing_balance_value = new_row_df.loc[:,new_row_df.columns == (row["Account To"])].iat[0,0]
                
                if row.Amount > ( old_grace_period_balance_value + old_interest_accruing_balance_value ):
                    logger.error("Attempting to make a credit card payment in excess of total debt. Exiting.")
                    raise ValueError
                    
                #payment is applied to this-cycle balance first.
                if row.Amount <= old_grace_period_balance_value:
                    new_grace_period_balance_value = old_grace_period_balance_value - row.Amount
                    new_interest_accruing_balance_value = old_interest_accruing_balance_value
                    
                #this includes where old_grace_period_balance_value is 0
                elif row.Amount > old_grace_period_balance_value:
                    new_grace_period_balance_value = 0
                    new_interest_accruing_balance_value = old_interest_accruing_balance_value + old_grace_period_balance_value - row.Amount
                
                else:
                    logger.error("Undefined case in satisfice while calculating credit card payment allocation.")
                    raise ValueError
                    
                new_row_df.loc[:,new_row_df.columns == (row["Account To"] + ' Grace Period Balance')] = new_grace_period_balance_value
                new_row_df.loc[:,new_row_df.columns == (row["Account To"])] = new_interest_accruing_balance_value

            #Income
            elif account_from__type == '' and account_to__type == 'Checking':
                new_row_df.loc[:,new_row_df.columns == (row["Account To"])] = new_row_df.loc[:,new_row_df.columns == (row["Account To"])] + row.Amount
            
            else:
                logger.info("row:")
                logger.info(row.to_string())
                old_from_value = float(new_row_df.loc[:,new_row_df.columns == (row["Account From"])].iat[0,0])
                new_from_value = old_from_value - float(row.Amount)
                
                #could put a log statement here
                
                new_row_df.loc[:,new_row_df.columns == (row["Account From"])] = new_from_value
                
            new_row_df.loc[:,"Memo"] = str(row.Memo) + ' ; ' + str(row.Memo)
            previous_rows_df = pd.concat([previous_rows_df,new_row_df])



                    
                #logger.info('Principal: '+str(old_principal_value)+' -> '+str(new_principal_value))
                #logger.info('Interest: '+str(old_interest_value)+' -> '+str(new_interest_value))
                    
            
            ##THIS LOGIC WORKED FOR THE OLD INPUT
#             assert sum(new_row_df.columns == row.Account) == 1 #assert there was a matching account
            
#             row.Amount = float(row.Amount)
            
#             #logger.info('Account Type:'+str(account_name_to_type_map[row.Account]))
#             #logger.info('row.Account:'+str(row.Account))
#             #logger.info('new_row_df:'+str(new_row_df))
#             if account_name_to_type_map[row.Account] == 'Loan':
#                 old_principal_value = float(new_row_df.loc[:,new_row_df.columns == row.Account].iat[0,0])
#                 old_interest_value = float(new_row_df.loc[:,new_row_df.columns == (row.Account+' Interest')].iat[0,0])
                
#                 #payment goes to interest only
#                 #logger.info('row.Amount*-1:'+str(row.Amount*-1)+' ?= '+str(old_interest_value))
#                 if row.Amount*-1 <= old_interest_value:
#                     #principal remains unchanged
#                     new_principal_value = old_principal_value
#                     new_row_df.loc[:,new_row_df.columns == row.Account]  = new_principal_value
                    
#                     new_interest_value = old_interest_value + row.Amount
#                     new_row_df.loc[:,new_row_df.columns == (row.Account+' Interest')] = new_interest_value
                    
#                 #this includes the case where interest = 0
#                 elif row.Amount*-1 > old_interest_value:
                    
#                     #principal remains unchanged
#                     new_principal_value = old_principal_value + old_interest_value + row.Amount
#                     new_row_df.loc[:,new_row_df.columns == row.Account] = new_principal_value
                    
#                     #Interest is now 0
#                     new_interest_value = 0
#                     new_row_df.loc[:,new_row_df.columns == (row.Account+' Interest')] = new_interest_value
                    
#                 else:
#                     logger.error('error in satisfice::loan interest payment calculation')
                    
#                 #logger.info('Principal: '+str(old_principal_value)+' -> '+str(new_principal_value))
#                 #logger.info('Interest: '+str(old_interest_value)+' -> '+str(new_interest_value))
                    
#             elif account_name_to_type_map[row.Account] == 'Credit':
#                 old_value = float(new_row_df.loc[:,new_row_df.columns == (row.Account + ' Grace Period Balance')].iat[0,0])
#                 new_value = old_value + float(row.Amount)

#                 logger.info(str(d)+' '+str(index)+' '+str(row.Memo)+' ; '+str(row.Account + ' Grace Period Balance')+' '+str(float(old_value))+' ('+str(float(row.Amount))+') -> '+str(float(new_value)))

#                 new_row_df.loc[:,new_row_df.columns == (row.Account + ' Grace Period Balance')] += row.Amount
#                 new_row_df.loc[:,"Memo"] = str(row.Memo) + ' ; ' + str(row.Memo)
            
#             else:
#                 old_value = float(new_row_df.loc[:,new_row_df.columns == row.Account].iat[0,0])
#                 new_value = old_value + float(row.Amount)

#                 logger.info(str(d)+' '+str(index)+' '+str(row.Memo)+' ; '+str(row.Account)+' '+str(float(old_value))+' ('+str(float(row.Amount))+') -> '+str(float(new_value)))

#                 new_row_df.loc[:,new_row_df.columns == row.Account] += row.Amount
#                 new_row_df.loc[:,"Memo"] = str(row.Memo) + ' ; ' + str(row.Memo)

#         previous_rows_df = pd.concat([previous_rows_df,new_row_df])
    satisficed_rows_df = previous_rows_df
    logger.info('exit satisfice()')
    return satisficed_rows_df


In [75]:
def main():
    parameter_tables_df = read_budget_item_excel_sheet('/home/hdickie/sandbox/Budget_Items.xlsx')
    
    
    parameter_tables_df[0]['Start Date'] = parameter_tables_df[0]['Start Date'].astype('string')
    
    accounts_df = parameter_tables_df[3]

    days_out = 90

    start_date_YYYYMMDD = datetime.datetime.now().strftime('%Y-%m-%d')
    scheduled_items_df = generate_series_from_item_schedule(parameter_tables_df[0],accounts_df
                                                            ,start_date_YYYYMMDD,days_out)

    logger.debug('scheduled_items_df:')
    logger.debug(scheduled_items_df.to_string())
    
    initial_balances_dict = get_initial_balances(accounts_df)

    current_ts_df = satisfice(start_date_YYYYMMDD,initial_balances_dict,scheduled_items_df,accounts_df,days_out)

    logger.debug('current_ts_df:')
    logger.debug(current_ts_df.to_string())
    
    #todo: more plz
    output_df = optimize_one_step(current_ts_df,proposed_transaction_df,accounts_df)
    #output_df = current_ts_df
    
    write_results(output_df)

In [76]:
def optimize(current_ts_df,scheduled_items_df,accounts_df_:
    logger.info('enter optimize())
             
    logger.info('exit optimize())         
    return (current_ts_df,[])
    #return (final_ts_df,finalized_dates_for_lower_priority_items_df)

def optimize_one_step(current_ts_df,proposed_transaction_df,accounts_df):
    logger.info('enter optimize_one_step()')
    logger.info('proposed_transaction_df:'+proposed_transaction_df.to_string())
    
    account_name_to_type_map = get_account_name_to_type_map(accounts_df)
    
    date_of_proposed_transaction = proposed_transaction_df.iloc[0,0]
    
    today_and_the_future_df = current_ts_df[current_ts_df['Date'] > date_of_proposed_transaction]

    checking_lookahead = min(today_and_the_future_df['Checking'])

    if row.Amount == 'FREE_BALANCE':
        #TODO: left off here
        pass

                
            
            #new_row_df = previous_rows_df.tail(1).copy()
            #new_row_df.loc[0,"Date"] = dpass
    
    return (updated_current_ts_df, executed_or_deferred_or_skipped_transaction_df)

In [77]:
initialize_logging(log_level="info")
main()

2022-06-03 15:12:39,283 - Expense_Forecast - INFO - enter satisfice()
2022-06-03 15:12:39,284 - Expense_Forecast - INFO - start_date_YYYYMMDD:2022-06-03
2022-06-03 15:12:39,285 - Expense_Forecast - INFO - initial_balances_dict:{'Checking': 3000.0, 'Savings': 0.0, 'Credit 1': 0.0, 'Credit 1 Grace Period Balance': 0.0, 'Loan A': 4746.18, 'Loan A Interest': 0.0, 'Loan B': 1919.55, 'Loan B Interest': 0.0, 'Loan C': 4726.68, 'Loan C Interest': 0.0, 'Loan D': 1823.31, 'Loan D Interest': 0.0, 'Loan E': 3359.17, 'Loan E Interest': 0.0}
2022-06-03 15:12:39,285 - Expense_Forecast - INFO - days_out:90
2022-06-03 15:12:39,288 - Expense_Forecast - INFO - cc_start_date_YYYYMMDD:2022-01-07
2022-06-03 15:12:39,289 - Expense_Forecast - INFO - cc_end_date_YYYYMMDD:2022-09-01
2022-06-03 15:12:39,290 - Expense_Forecast - INFO - d:2022-06-03
2022-06-03 15:12:39,291 - Expense_Forecast - INFO - d:2022-06-04
2022-06-03 15:12:39,298 - Expense_Forecast - INFO - row.Amount:30.0
2022-06-03 15:12:39,299 - Expense_